# 02 · Entrenamiento de modelos

Entrada: `datos_cleveland_limpios.csv`.
Salida: `modelos_tfm.joblib` (modelos, umbrales, partición) y `resumen_entrenamiento.csv`.

In [ ]:
import glob
import numpy as np
import pandas as pd
import joblib
import sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

SEMILLA = 42
print("scikit-learn", sklearn.__version__)

In [ ]:
try:
    from google.colab import files
    if not glob.glob("datos_cleveland_limpios*.csv"):
        files.upload()
except ImportError:
    pass

df = pd.read_csv(sorted(glob.glob("datos_cleveland_limpios*.csv"))[0])
print(df.shape, "| eventos:", df["enfermedad"].sum())

## 1. Variables y partición

In [ ]:
VARS_DF       = ["age", "sex", "cp"]
VARS_CONSULTA = VARS_DF + ["trestbps", "chol", "fbs", "restecg_anormal"]
VARS_COMPLETO = VARS_CONSULTA + ["thalach", "exang", "oldpeak", "slope", "ca", "thal"]

CONTINUAS = ["age", "trestbps", "chol", "thalach", "oldpeak", "ca"]
NOMINALES = ["cp", "slope", "thal"]
BINARIAS  = ["sex", "fbs", "restecg_anormal", "exang"]

X = df.drop(columns=["num", "enfermedad", "restecg"])
y = df["enfermedad"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEMILLA)

print(f"Entrenamiento: {len(X_train)} ({y_train.sum()} eventos) | Test: {len(X_test)} ({y_test.sum()} eventos)")
print("Perdidos train/test:", X_train[["ca", "thal"]].isna().sum().sum(), X_test[["ca", "thal"]].isna().sum().sum())

## 2. Preprocesado y ajuste

In [ ]:
def preprocesador(variables):
    cont = [v for v in CONTINUAS if v in variables]
    nom  = [v for v in NOMINALES if v in variables]
    bina = [v for v in BINARIAS  if v in variables]
    pasos = []
    if cont:
        pasos.append(("continuas", Pipeline([
            ("imputar", SimpleImputer(strategy="most_frequent")),
            ("escalar", StandardScaler())]), cont))
    if nom:
        pasos.append(("nominales", Pipeline([
            ("imputar", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))]), nom))
    if bina:
        pasos.append(("binarias", "passthrough", bina))
    return ColumnTransformer(pasos)


CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
REJILLA_LR = {"modelo__C": [0.001, 0.01, 0.1, 1, 10, 100]}
REJILLA_RF = {"modelo__n_estimators": [300],
              "modelo__max_depth": [3, 5, None],
              "modelo__min_samples_leaf": [1, 5, 10],
              "modelo__max_features": ["sqrt"]}


def entrenar(nombre, variables, modelo, rejilla):
    pipe = Pipeline([("prep", preprocesador(variables)), ("modelo", modelo)])
    if rejilla:
        busqueda = GridSearchCV(pipe, rejilla, cv=CV, scoring="roc_auc", n_jobs=-1)
        busqueda.fit(X_train[variables], y_train)
        mejor, params, auc_cv = busqueda.best_estimator_, busqueda.best_params_, busqueda.best_score_
        sd_cv = busqueda.cv_results_["std_test_score"][busqueda.best_index_]
    else:
        aucs = [roc_auc_score(y_train.iloc[te],
                              pipe.fit(X_train[variables].iloc[tr], y_train.iloc[tr])
                                  .predict_proba(X_train[variables].iloc[te])[:, 1])
                for tr, te in CV.split(X_train, y_train)]
        mejor, params, auc_cv, sd_cv = pipe.fit(X_train[variables], y_train), {}, np.mean(aucs), np.std(aucs)
    print(f"{nombre:<28} AUC CV = {auc_cv:.3f} ± {sd_cv:.3f}  {params}")
    return {"modelo": mejor, "variables": variables, "params": params,
            "auc_cv": auc_cv, "sd_cv": sd_cv}

In [ ]:
ESPECIFICACIONES = [
    ("Logística DF reentrenada", VARS_DF,       LogisticRegression(C=1e6, max_iter=1000), None),
    ("Logística consulta",       VARS_CONSULTA, LogisticRegression(max_iter=1000), REJILLA_LR),
    ("Logística completa",       VARS_COMPLETO, LogisticRegression(max_iter=1000), REJILLA_LR),
    ("Random Forest consulta",   VARS_CONSULTA, RandomForestClassifier(random_state=SEMILLA), REJILLA_RF),
    ("Random Forest completo",   VARS_COMPLETO, RandomForestClassifier(random_state=SEMILLA), REJILLA_RF),
]
modelos = {nombre: entrenar(nombre, v, m, r) for nombre, v, m, r in ESPECIFICACIONES}

## 3. Diamond-Forrester (tabla publicada)

Tabla 3 para pacientes sintomáticos; tabla 2 (prevalencia en autopsias) para asintomáticos.

In [ ]:
# Claves (cp, sexo); valores en % para las décadas 30-39, 40-49, 50-59, 60-69
TABLA_DF = {
    (3, 1): [5.2, 14.1, 21.5, 28.1],  (3, 0): [0.8, 2.8, 8.4, 18.6],
    (2, 1): [21.8, 46.1, 58.9, 67.1], (2, 0): [4.2, 13.3, 32.4, 54.4],
    (1, 1): [69.7, 87.3, 92.0, 94.3], (1, 0): [25.8, 55.2, 79.4, 90.6],
    (4, 1): [1.9, 5.5, 9.7, 12.3],    (4, 0): [0.3, 1.0, 3.2, 7.5],
}

def prob_diamond_forrester(datos):
    decada = ((datos["age"] // 10) - 3).clip(0, 3).astype(int)
    return np.array([TABLA_DF[(int(cp), int(sx))][d] / 100
                     for cp, sx, d in zip(datos["cp"], datos["sex"], decada)])

prob_diamond_forrester(pd.DataFrame({"age": [55, 72], "sex": [1, 0], "cp": [1, 4]}))  # [0.92, 0.075]

## 4. Umbral de descarte (sensibilidad ≥ 90 %, predicciones fuera de pliegue)

In [ ]:
def umbral_sensibilidad(y_real, prob, objetivo=0.90):
    candidatos = np.sort(np.unique(prob))[::-1]
    for u in candidatos:
        if ((prob >= u) & (y_real == 1)).sum() / (y_real == 1).sum() >= objetivo:
            return float(u)
    return float(candidatos[-1])

umbrales = {}
for nombre, m in modelos.items():
    prob_oof = cross_val_predict(m["modelo"], X_train[m["variables"]], y_train,
                                 cv=CV, method="predict_proba")[:, 1]
    umbrales[nombre] = umbral_sensibilidad(y_train.values, prob_oof)
umbrales["Diamond-Forrester (tabla)"] = umbral_sensibilidad(y_train.values, prob_diamond_forrester(X_train))
pd.Series(umbrales, name="Umbral").round(3)

## 5. Exportación

In [ ]:
resumen = pd.DataFrame({
    "Variables": {k: len(v["variables"]) for k, v in modelos.items()},
    "AUC CV (media)": {k: round(v["auc_cv"], 3) for k, v in modelos.items()},
    "AUC CV (DE)": {k: round(v["sd_cv"], 3) for k, v in modelos.items()},
    "Hiperparámetros": {k: v["params"] for k, v in modelos.items()},
})
resumen.to_csv("resumen_entrenamiento.csv")

joblib.dump({"modelos": modelos, "umbrales": umbrales, "tabla_df": TABLA_DF,
             "indices_train": X_train.index.tolist(), "indices_test": X_test.index.tolist(),
             "semilla": SEMILLA, "sklearn": sklearn.__version__}, "modelos_tfm.joblib")
try:
    from google.colab import files
    files.download("modelos_tfm.joblib")
    files.download("resumen_entrenamiento.csv")
except ImportError:
    pass
resumen